Projet : Analyse des Offres d'Emploi LinkedIn avec Snowflake

In [ ]:
CREATE DATABASE IF NOT EXISTS LINKEDIN;
CREATE SCHEMA IF NOT EXISTS LINKEDIN.RAW;
USE SCHEMA LINKEDIN.RAW;

CREATE OR REPLACE STAGE LINKEDIN_STAGE
  URL = 's3://snowflake-lab-bucket/';

Nous créons ici la base de données LINKEDIN, le schéma RAW pour stocker les données brutes, ainsi qu'un stage externe pointant vers le bucket S3 contenant les fichiers LinkedIn.

In [ ]:
CREATE OR REPLACE FILE FORMAT csv_format
  TYPE                        = 'CSV'
  FIELD_OPTIONALLY_ENCLOSED_BY = '"'
  SKIP_HEADER                 = 1
  NULL_IF                     = ('NULL', 'null', '')
  EMPTY_FIELD_AS_NULL         = TRUE;

CREATE OR REPLACE FILE FORMAT json_format
  TYPE              = 'JSON'
  STRIP_OUTER_ARRAY = TRUE;

Nous définissons ici deux formats de fichiers : un pour les CSV qui gère les guillemets et les valeurs nulles, et un pour les JSON qui permet de lire chaque objet du tableau individuellement.

In [ ]:
USE SCHEMA LINKEDIN.RAW;

CREATE OR REPLACE TABLE job_postings (
  job_id                     VARCHAR,
  company_name               VARCHAR,
  title                      VARCHAR,
  description                TEXT,
  max_salary                 VARCHAR,
  med_salary                 VARCHAR,
  min_salary                 VARCHAR,
  pay_period                 VARCHAR,
  formatted_work_type        VARCHAR,
  location                   VARCHAR,
  applies                    VARCHAR,
  original_listed_time       VARCHAR,
  remote_allowed             VARCHAR,
  views                      VARCHAR,
  job_posting_url            VARCHAR,
  application_url            VARCHAR,
  application_type           VARCHAR,
  expiry                     VARCHAR,
  closed_time                VARCHAR,
  formatted_experience_level VARCHAR,
  skills_desc                TEXT,
  listed_time                VARCHAR,
  posting_domain             VARCHAR,
  sponsored                  VARCHAR,
  work_type                  VARCHAR,
  currency                   VARCHAR,
  compensation_type          VARCHAR
);

CREATE OR REPLACE TABLE benefits (
  job_id   VARCHAR,
  inferred VARCHAR,
  type     VARCHAR
);

CREATE OR REPLACE TABLE companies (
  company_id   VARCHAR,
  name         VARCHAR,
  description  TEXT,
  company_size VARCHAR,
  state        VARCHAR,
  country      VARCHAR,
  city         VARCHAR,
  zip_code     VARCHAR,
  address      VARCHAR,
  url          VARCHAR
);

CREATE OR REPLACE TABLE employee_counts (
  company_id     VARCHAR,
  employee_count VARCHAR,
  follower_count VARCHAR,
  time_recorded  VARCHAR
);

CREATE OR REPLACE TABLE job_skills (
  job_id    VARCHAR,
  skill_abr VARCHAR
);

CREATE OR REPLACE TABLE job_industries (
  job_id      VARCHAR,
  industry_id VARCHAR
);

CREATE OR REPLACE TABLE company_industries (
  company_id VARCHAR,
  industry   VARCHAR
);

CREATE OR REPLACE TABLE company_specialities (
  company_id VARCHAR,
  speciality VARCHAR
);

In [ ]:
USE SCHEMA LINKEDIN.RAW;

COPY INTO job_postings
FROM @linkedin_stage/job_postings.csv
FILE_FORMAT = csv_format
ON_ERROR = 'CONTINUE'
FORCE = TRUE;

COPY INTO benefits
FROM @linkedin_stage/benefits.csv
FILE_FORMAT = csv_format
ON_ERROR = 'CONTINUE'
FORCE = TRUE;

COPY INTO employee_counts
FROM @linkedin_stage/employee_counts.csv
FILE_FORMAT = csv_format
ON_ERROR = 'CONTINUE'
FORCE = TRUE;

COPY INTO job_skills
FROM @linkedin_stage/job_skills.csv
FILE_FORMAT = csv_format
ON_ERROR = 'CONTINUE'
FORCE = TRUE;

Nous chargeons ici les données depuis le bucket S3 vers les tables Snowflake à l'aide de la commande COPY INTO. Les fichiers CSV sont importés directement dans leurs tables respectives.

In [ ]:
COPY INTO companies
FROM (
  SELECT
    $1:company_id::VARCHAR,
    $1:name::VARCHAR,
    $1:description::VARCHAR,
    $1:company_size::VARCHAR,
    $1:state::VARCHAR,
    $1:country::VARCHAR,
    $1:city::VARCHAR,
    $1:zip_code::VARCHAR,
    $1:address::VARCHAR,
    $1:url::VARCHAR
  FROM @linkedin_stage/companies.json
)
FILE_FORMAT = json_format
ON_ERROR = 'CONTINUE'
FORCE = TRUE;

COPY INTO company_industries
FROM (
  SELECT $1:company_id::VARCHAR, $1:industry::VARCHAR
  FROM @linkedin_stage/company_industries.json
)
FILE_FORMAT = json_format
ON_ERROR = 'CONTINUE'
FORCE = TRUE;

COPY INTO company_specialities
FROM (
  SELECT $1:company_id::VARCHAR, $1:speciality::VARCHAR
  FROM @linkedin_stage/company_specialities.json
)
FILE_FORMAT = json_format
ON_ERROR = 'CONTINUE'
FORCE = TRUE;

COPY INTO job_industries
FROM (
  SELECT $1:job_id::VARCHAR, $1:industry_id::VARCHAR
  FROM @linkedin_stage/job_industries.json
)
FILE_FORMAT = json_format
ON_ERROR = 'CONTINUE'
FORCE = TRUE;

Nous chargeons ici les fichiers JSON depuis le bucket S3. Contrairement aux CSV, les JSON nécessitent d'extraire chaque champ manuellement avec la notation $1.

**Analyses**

Analyse 1 : Top 10 des titres les plus publiés par industrie :

In [ ]:
SELECT
  ci.industry                                                          AS industrie,
  jp.title                                                             AS titre,
  COUNT(*)                                                             AS nb_offres,
  ROW_NUMBER() OVER (PARTITION BY ci.industry ORDER BY COUNT(*) DESC) AS classement
FROM job_postings jp
JOIN companies c           ON SPLIT_PART(jp.company_name, '.', 1) = c.company_id
JOIN company_industries ci ON c.company_id = ci.company_id
GROUP BY ci.industry, jp.title
QUALIFY classement <= 10
ORDER BY ci.industry, nb_offres DESC;

Résultat : Dans le secteur Comptabilité, le poste 'Tax Manager, Family Office' arrive en tête avec 72 offres publiées, suivi des postes de Tax Intern et Tax Staff avec 70 offres chacun. Cela reflète une forte demande dans les métiers de la fiscalité et de la comptabilité


Lors de l'exploration des données, nous avons constaté que la colonne company_name dans job_postings contient en réalité un identifiant numérique au format 24601924.0 et non un nom d'entreprise. Nous utilisons donc SPLIT_PART pour supprimer le .0 et faire correspondre cet identifiant avec le company_id de la table companies, qui nous permet ensuite de récupérer le secteur d'activité via company_industries.

Analyse 2 : Top 10 des postes les mieux rémunérés par industrie :

In [ ]:
SELECT
  ci.industry                                                                     AS industrie,
  jp.title                                                                        AS titre,
  ROUND(AVG(TRY_CAST(jp.max_salary AS FLOAT)), 0)                                AS salaire_max_moyen,
  ROW_NUMBER() OVER (PARTITION BY ci.industry ORDER BY AVG(TRY_CAST(jp.max_salary AS FLOAT)) DESC) AS classement
FROM job_postings jp
JOIN companies c           ON SPLIT_PART(jp.company_name, '.', 1) = c.company_id
JOIN company_industries ci ON c.company_id = ci.company_id
WHERE jp.max_salary IS NOT NULL AND jp.max_salary != ''
GROUP BY ci.industry, jp.title
QUALIFY classement <= 10
ORDER BY ci.industry, salaire_max_moyen DESC;

Cette analyse identifie les postes les mieux rémunérés par secteur d'activité. On calcule la moyenne du salaire maximum par titre de poste, ce qui permet de comparer les niveaux de rémunération entre les différentes industries. 
Résultat : Dans le secteur Accounting, le Senior Manager en fiscalité internationale arrive en tête avec un salaire maximum moyen de 303 100, suivi du Manager en Risk & Controls Consulting à 223 100. Cela confirme que les postes de management senior sont les mieux rémunérés.

Analyse 3 : Répartition par taille d'entreprise


In [ ]:

SELECT
  CASE c.company_size
    WHEN '0' THEN 'Solo'
    WHEN '1' THEN '1-10 employés'
    WHEN '2' THEN '11-50 employés'
    WHEN '3' THEN '51-200 employés'
    WHEN '4' THEN '201-500 employés'
    WHEN '5' THEN '501-1000 employés'
    WHEN '6' THEN '1001-5000 employés'
    WHEN '7' THEN '5000+ employés'
  END AS taille,
  COUNT(jp.job_id) AS nb_offres
FROM job_postings jp
JOIN companies c ON SPLIT_PART(jp.company_name, '.', 1) = c.company_id
GROUP BY c.company_size, taille
ORDER BY c.company_size;

Cette analyse montre la répartition des offres selon la taille des entreprises. 
Résultat : Les très grandes entreprises de 5000+ employés publient le plus d'offres avec 5 127 postes, suivies des entreprises de 501-1000 employés avec 2 794 offres. Les très petites structures de 1 à 10 employés restent les moins actifs avec seulement 1 124 offres.

Analyse 4 : Répartition par secteur d'activité

In [ ]:
SELECT
  ci.industry       AS secteur,
  COUNT(DISTINCT jp.job_id) AS nb_offres
FROM job_postings jp
JOIN companies c           ON SPLIT_PART(jp.company_name, '.', 1) = c.company_id
JOIN company_industries ci ON c.company_id = ci.company_id
GROUP BY ci.industry
ORDER BY nb_offres DESC
LIMIT 20;

Le secteur Staffing & Recruiting domine avec 2 240 offres, suivi de l'IT & Services et de la santé. Cela reflète une forte demande en recrutement et en technologie sur le marché de l'emploi LinkedIn.


Analyse 5 : Répartition par type d'emploi

In [ ]:
SELECT
  formatted_work_type                                        AS type_emploi,
  COUNT(*)                                                   AS nb_offres,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1)        AS pourcentage
FROM job_postings
WHERE formatted_work_type IS NOT NULL
AND formatted_work_type != ''
GROUP BY formatted_work_type
ORDER BY nb_offres DESC;

Les offres Full-time représentent 80.9% des publications, ce qui confirme que LinkedIn est principalement utilisé pour le recrutement de postes permanents. Les contrats et temps partiels restent minoritaires avec respectivement 10.9% et 6.4%.